**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Estimation Theory

The missing link between [Random Variables](../Analysis/Random_Variables.ipynb) and every filter/model in this curriculum: given noisy data, what is the *best* guess of the underlying parameter — and what does 'best' even mean? Four sessions: maximum likelihood, the Cramér–Rao floor no estimator can beat, Bayesian estimation, and sufficiency.

## 0. Introduction

Setup: data $x_1, \dots, x_n \sim p(x; \theta)$, unknown parameter $\theta$. An *estimator* $\hat{\theta}(x_{1:n})$ is any function of the data — the question is which ones are good, judged by **bias** $E[\hat\theta] - \theta$ and **variance**.

## 1. Pre-requisites

- [Random Variables](../Analysis/Random_Variables.ipynb) — densities, expectation, LOTUS.
- [Optimization](../Optimization/Optimization.ipynb) S1–S2 — we'll maximize likelihoods.
- [Independence](../Analysis/Independence.ipynb) — i.i.d. sampling and the LLN.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 4 — *Maximum Likelihood* (~35 min)
**Goal:** derive MLE for the Gaussian; understand log-likelihood as the natural loss.
**Builds on:** [Random Variables](../Analysis/Random_Variables.ipynb). &nbsp; **Feeds into:** Session 2 (CRLB).

---

<details>
<summary>🎓 <b>Teacher notes — Session 1: Maximum Likelihood</b></summary>

**Timing (~35 min).** 10 min the flip · 10 min the Gaussian derivation · 8 min the landscape demo · 7 min why MLE is biased here.

**Board first — the flip is the whole idea.** A density answers "given $\theta$, how probable is this data?" MLE reverses the question: "given the data I actually saw, which $\theta$ makes it least surprising?" Same function, read along the other argument. Emphasise that the likelihood is **not a probability distribution over $\theta$** — it does not integrate to 1 in $\theta$, and treating it as one is the mistake Session 3 fixes properly with a prior.

**Then the observation that pays for the whole workshop.** Taking logs turns the i.i.d. product into a sum without moving the argmax — and *that sum is where machine learning's loss functions come from*. Least squares **is** Gaussian MLE; cross-entropy **is** categorical MLE. Ask the room where they have seen $\sum_i (x_i - \mu)^2$ before; deriving it from a Gaussian likelihood rather than assuming it is the moment the session earns its place in the curriculum.

**Do the derivation, both parameters.** $\partial_\mu$ gives $\hat\mu = \bar x$ immediately, and note aloud that maximising a Gaussian likelihood in $\mu$ *is* minimising squared error — same equation. Then $\partial_{\sigma^2}$ gives the $1/n$ variance rather than $1/(n-1)$.

**Do not skip the bias, and give it a reason rather than a correction factor.** The MLE variance is biased low by $\frac{n-1}{n}$ because it measures spread around $\bar x$, which is itself fitted to the data and therefore sits closer to the samples than the true $\mu$ does. One degree of freedom was spent estimating the mean. Students who learn "divide by $n-1$" as a rule find this genuinely clarifying.

**Then the honest framing of MLE's guarantees.** MLE is **not** automatically unbiased — this is the counterexample. What it *is*: consistent (converges to truth as $n\to\infty$), asymptotically efficient (attains the Session 2 bound in the limit), and invariant under reparameterisation. Ask which property matters most in practice; for large $n$ it is efficiency, which is why MLE is the default despite the bias.

**On the demo.** The likelihood landscape is a contour plot with the peak marked — worth pointing out that the peak is at $(\bar x, \hat\sigma)$ and that it is *near* but not *at* the truth, which is sampling variability rather than a defect. Note also the surface's shape: elongated along $\sigma$ near the peak, meaning $\sigma$ is less sharply determined than $\mu$ by 40 samples. That curvature is literally Fisher information, and it is Session 2's subject — flag it here so the connection is ready.
</details>

## 2. The Maximum Likelihood Principle

💡 **Intuition.** Flip the density around: instead of 'given $\theta$, how probable is data $x$?', ask 'given the data I *saw*, which $\theta$ would have made it least surprising?' The likelihood $L(\theta) = \prod_i p(x_i; \theta)$ scores each candidate; MLE picks the top. Taking logs turns the product into a sum (i.i.d.!) without moving the argmax — and that log-likelihood sum is where nearly every ML loss function comes from: least squares *is* Gaussian MLE, cross-entropy *is* categorical MLE.

### Derivation: Gaussian mean and variance

For $x_i \sim \mathcal{N}(\mu, \sigma^2)$:
$$\log L = -\frac{n}{2}\log(2\pi\sigma^2) - \frac{1}{2\sigma^2}\sum_i (x_i - \mu)^2$$
$\partial_\mu \log L = \frac{1}{\sigma^2}\sum_i (x_i - \mu) = 0 \Rightarrow \hat\mu = \bar{x}$ — the sample mean, and note: maximizing the Gaussian likelihood in $\mu$ *is* minimizing squared error.

$\partial_{\sigma^2} \log L = 0 \Rightarrow \hat{\sigma}^2 = \frac{1}{n}\sum_i (x_i - \bar{x})^2$ — biased by the factor $\frac{n-1}{n}$ (it 'spends' one data point estimating $\mu$ first). MLE is not automatically unbiased; it *is* consistent and asymptotically optimal.

In [2]:
# Likelihood surface for a small Gaussian sample — the argmax IS the estimate
x = rng.normal(2.0, 1.5, size=40)
mus = np.linspace(0.5, 3.5, 200)
sigmas = np.linspace(0.7, 3.0, 200)
MU, SG = np.meshgrid(mus, sigmas)
ll = -len(x)/2*np.log(2*np.pi*SG**2) - ((x[:,None,None] - MU)**2).sum(0) / (2*SG**2)

plt.figure(figsize=(6, 3.6))
plt.contourf(MU, SG, ll, levels=40)
plt.colorbar(label="log-likelihood")
plt.plot(x.mean(), x.std(), "r*", markersize=14, label="MLE $(\\bar{x}, \\hat{\\sigma})$")
plt.plot(2.0, 1.5, "wo", label="truth")
plt.xlabel("μ"); plt.ylabel("σ"); plt.legend()
plt.title("The likelihood landscape: MLE = its peak")
plt.tight_layout(); plt.show()

/tmp/ipykernel_2014894/514868749.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** The log-likelihood over the $(\mu, \sigma)$ plane, with the MLE marked at its peak and the truth marked separately. Two things to read off it.

**The estimate is near the truth but not on it**, and that is sampling variability rather than a flaw. With 40 samples the peak lands close; with 4 it would wander noticeably; with 4000 it would be nearly indistinguishable. MLE is *consistent* — it converges to truth as $n$ grows — which is a different and weaker promise than being right for any particular sample.

**Look at the shape of the surface, because it is Session 2's subject in disguise.** The contours are noticeably elongated along the $\sigma$ direction near the peak: moving $\sigma$ changes the log-likelihood less than moving $\mu$ does. That means $\sigma$ is **less sharply determined** by this data than $\mu$ is. The curvature of the log-likelihood at its peak literally *is* Fisher information, so a flat direction means little information and a high-variance estimate. You can read an estimator's precision off the geometry before computing any variance.

**And the reframing that makes this workshop matter to everything downstream.** Taking logs turned the i.i.d. product $\prod_i p(x_i;\theta)$ into the sum plotted here — and that sum is where machine learning's loss functions come from. For a Gaussian,
$$\log L = -\frac{n}{2}\log(2\pi\sigma^2) - \frac{1}{2\sigma^2}\sum_i (x_i-\mu)^2,$$
so maximising the likelihood in $\mu$ **is** minimising $\sum_i(x_i-\mu)^2$. Least squares is not a convenient choice of loss that happens to work; it is Gaussian maximum likelihood. Cross-entropy is categorical MLE by the same route. Every time you write `MSELoss`, you are asserting a Gaussian noise model.

**One caution about what the likelihood is not.** It is not a probability distribution over $\theta$ — it does not integrate to 1 along that axis, and the height of the peak carries no meaning on its own. Treating it as a belief about $\theta$ requires a prior, which is exactly the step Session 3 takes.

**And the MLE for $\sigma^2$ shown here is biased**, by the factor $\frac{n-1}{n}$. The reason is worth having: it measures spread around $\bar x$, which was itself fitted to this data and therefore sits closer to the samples than the true $\mu$ does. One degree of freedom went into estimating the mean. That is where "divide by $n-1$" comes from — not a correction bolted on, but the consequence of having already spent a parameter.

---
### 🕐 Session 2 of 4 — *Bias, Variance & the Cramér–Rao Bound* (~40 min)
**Goal:** prove the variance floor; check estimators against it.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (Bayes).

---

<details>
<summary>🎓 <b>Teacher notes — Session 2: Bias, Variance & the Cramér–Rao Bound</b></summary>

**Timing (~40 min).** 10 min the sensitivity intuition · 12 min Fisher information and the proof · 10 min the Monte Carlo · 8 min what the bound does not cover.

**Board first — build the intuition before the formula.** How well *could* any estimator possibly do? It depends on how much the data's distribution **moves** when $\theta$ moves. If wiggling $\theta$ barely changes $p(x;\theta)$, then the data barely distinguishes nearby $\theta$ values and no amount of cleverness recovers what is not there. Fisher information measures exactly that sensitivity. Framing it as "how loudly does the data respond to $\theta$?" makes $I(\theta) = E[(\partial_\theta \log p)^2]$ a measurement rather than a definition.

**Connect it to Session 1's picture.** Fisher information is the expected curvature of the log-likelihood at its peak. A sharply peaked likelihood means high information and a precise estimate; a flat one means low information. The elongated $\sigma$ direction in Session 1's contour plot was low information, visible before any variance was computed. That connection is worth making explicitly — it turns two sessions into one idea.

**The proof is short and worth doing.** Differentiate the unbiasedness identity $\int \hat\theta\, p\, dx = \theta$ in $\theta$ to get $\mathrm{Cov}(\hat\theta, s) = 1$, then apply Cauchy–Schwarz: $1 \le \mathrm{Var}(\hat\theta)\mathrm{Var}(s) = \mathrm{Var}(\hat\theta)\, nI(\theta)$. Point out that this is [Hilbert Spaces](../Hilbert_Spaces/Hilbert_Spaces.ipynb)' Cauchy–Schwarz doing load-bearing work for the third or fourth time in this curriculum — the same inequality that bounds correlation and proves the matched filter optimal.

**Set the demo up as a competition, and have the room predict.** Three estimators of a Gaussian mean. Most will expect the median to win on robustness grounds, and it is worth letting them say so before the numbers appear: mean 0.0900, median 0.1402, midrange 0.2999, against a CRLB of 0.0900. The mean sits **exactly** on the floor.

**Then give the theoretical values, because they turn a demo into a verification.** The median's asymptotic variance for Gaussian data is $\pi\sigma^2/2n = 0.1414$, against a measured 0.1402 — so the median is about **64% efficient**, meaning it needs roughly 1.57× as many samples to match the mean. That is a concrete price, not a vague "worse."

**The most important caveat is in the printed line — do not let it pass as a throwaway.** "For Gaussian data!" The median's inefficiency is entirely conditional on Gaussianity. Under heavy tails, or with a few outliers, the median wins decisively and the mean can be arbitrarily bad. Ask what changes: CRLB is computed *for an assumed model*, so it is a floor within that model and says nothing about performance when the model is wrong. Optimality is always relative to assumptions, and this is the cleanest place in the curriculum to say so.

**Two further limits worth naming.** CRLB applies to *unbiased* estimators — biased ones can and do beat it in mean-squared error, which is exactly what regularisation and shrinkage exploit. And it requires regularity conditions that fail for some problems (e.g. a uniform distribution's endpoint). It is a powerful bound, not a universal one.
</details>

## 3. The Cramér–Rao Lower Bound

💡 **Intuition.** How well *could* any unbiased estimator possibly do? It depends on how much the data's distribution *moves* when $\theta$ moves. If wiggling $\theta$ barely changes $p(x;\theta)$, the data barely carries information about $\theta$, and no cleverness can recover it. **Fisher information** $I(\theta)$ quantifies that sensitivity, and CRLB says: variance $\ge 1/(n I(\theta))$. It's the thermodynamic limit of estimation — the reference line every tracking paper plots.

**Definition.** $I(\theta) = E\big[ (\partial_\theta \log p(x;\theta))^2 \big]$ (the *score*'s variance; the score has mean zero).

**Theorem (CRLB).** For unbiased $\hat\theta$ from $n$ i.i.d. samples (regularity assumed): $\mathrm{Var}(\hat\theta) \ge \frac{1}{n I(\theta)}$.

**Proof sketch.** Unbiasedness $\int \hat\theta \, p \, dx = \theta$; differentiate both sides in $\theta$ to get $\mathrm{Cov}(\hat\theta, s) = 1$ where $s$ is the total score. Cauchy–Schwarz ([Measure Theory](../Analysis/Measure_Theory.ipynb)'s Hölder with $p = q = 2$) then gives $1 \le \mathrm{Var}(\hat\theta) \, \mathrm{Var}(s) = \mathrm{Var}(\hat\theta) \, n I(\theta)$. $\blacksquare$

**Example.** Gaussian mean: $\log p = -(x-\mu)^2/2\sigma^2 + c$, score $= (x-\mu)/\sigma^2$, so $I = 1/\sigma^2$ and CRLB $= \sigma^2/n$ — *exactly* the variance of $\bar{x}$. The sample mean is **efficient**: it sits on the floor.

In [3]:
# Monte Carlo: three estimators of a Gaussian mean vs the CRLB
n, trials, sigma = 25, 20000, 1.5
X = rng.normal(2.0, sigma, size=(trials, n))
crlb = sigma**2 / n

for name, est in [("sample mean", X.mean(1)),
                  ("sample median", np.median(X, 1)),
                  ("midrange", (X.max(1) + X.min(1)) / 2)]:
    print(f"{name:14s} variance {est.var():.4f}   (CRLB {crlb:.4f})")
print("→ the mean achieves the bound; the others pay a premium (for Gaussian data!)")

sample mean    variance 0.0900   (CRLB 0.0900)
sample median  variance 0.1402   (CRLB 0.0900)
midrange       variance 0.2999   (CRLB 0.0900)
→ the mean achieves the bound; the others pay a premium (for Gaussian data!)


**What just happened.** Three estimators of the same Gaussian mean, 20,000 trials each, measured against a theoretical floor:

| estimator | variance | vs CRLB | efficiency |
|---|---|---|---|
| sample mean | **0.0900** | **1.00×** | 100% |
| sample median | 0.1402 | 1.56× | ~64% |
| midrange | 0.2999 | 3.33× | ~30% |

**The sample mean sits exactly on the floor**, matching CRLB $= \sigma^2/n = 0.0900$ to four decimals. It is *efficient*: no unbiased estimator of a Gaussian mean can do better, ever, by any method. That is a strong statement — not "the best we know of" but "the best that exists."

**And the other two are quantifiably, not vaguely, worse.** The median's asymptotic variance for Gaussian data is $\pi\sigma^2/2n = 0.1414$, against the measured 0.1402 — theory confirmed. Its **64% efficiency** converts into a concrete price: the median needs about **1.57× as many samples** to match the mean's precision. If samples cost money, that is a budget line.

**Now the caveat in the printed output, which is the most important sentence in the session.** *"for Gaussian data!"* Every ranking above is conditional on Gaussianity. Change the distribution and the ordering changes:

- With **heavy tails** (Cauchy, say) the sample mean's variance is infinite and the median wins outright.
- With a **few outliers** — one mis-recorded sensor reading — the mean is dragged arbitrarily far while the median barely moves.

So the median is not a worse estimator. It is a *less efficient* estimator **for this model** and a far more *robust* one for models with contamination, and which property you want is an engineering decision about your data rather than a mathematical fact.

That generalises into the caution to carry away: **CRLB is a floor computed for an assumed model.** It tells you the best achievable performance *if your model is right*, and says nothing whatsoever about what happens when it is wrong. Optimality is always relative to assumptions — a theme that recurs in [Statistical SP](../../Intro_DSP/Statistical_Signal_Processing.ipynb)'s matched filter (optimal for known signal in white Gaussian noise) and in Wiener filtering (optimal given correct PSDs).

**Two further limits on the bound itself.** It applies only to **unbiased** estimators — biased ones routinely beat it in mean-squared error, which is precisely what shrinkage and regularisation exploit, and why ridge regression can outperform ordinary least squares. And it requires regularity conditions that genuinely fail for some problems, such as estimating the endpoint of a uniform distribution. CRLB is powerful and it is not universal.

**Why Fisher information is the right currency.** $I(\theta) = E[(\partial_\theta \log p)^2]$ measures how much the data's distribution *moves* when $\theta$ moves. If wiggling $\theta$ barely changes $p(x;\theta)$, the data cannot distinguish nearby values and no estimator can recover what is not there. It is also the expected curvature of the log-likelihood at its peak — so Session 1's elongated contour along $\sigma$ was low Fisher information, showing an imprecise estimate before any variance was computed.

---
### 🕐 Session 3 of 4 — *Bayesian Estimation* (~35 min)
**Goal:** treat θ as random; derive MMSE = posterior mean; connect to Kalman.
**Builds on:** Session 2. &nbsp; **Feeds into:** Session 4 (sufficiency).

---

<details>
<summary>🎓 <b>Teacher notes — Session 3: Bayesian Estimation</b></summary>

**Timing (~35 min).** 8 min the philosophical shift · 10 min MMSE = posterior mean · 10 min the precision-weighted blend · 7 min the Kalman connection.

**Board first — state the shift precisely, and avoid the tribal argument.** The frequentist treats $\theta$ as fixed-but-unknown and asks what would happen over repeated experiments. The Bayesian treats $\theta$ as *random*, carries a belief about it, and updates that belief with data. Present this as two different questions rather than two camps: "what would this procedure do in repetition?" and "what should I believe given what I saw?" are both legitimate, and they have different answers.

**MMSE = posterior mean is the session's theorem.** Minimising $E[(\hat\theta-\theta)^2]$ over *all* functions of the data gives $\hat\theta = E[\theta \mid x]$. The proof is one line — expand the square around the conditional mean and the cross term vanishes by iterated expectation — and it is worth doing because the result is stronger than students expect: not the best linear estimator, the best estimator *of any form*.

**The blend formula is the thing to put on the board and leave there.**
$$E[\theta\mid x] = \frac{\tau^2}{\tau^2 + \sigma^2/n}\bar x + \frac{\sigma^2/n}{\tau^2+\sigma^2/n}\mu_0$$
Read it aloud as a **precision-weighted average**: each source contributes in proportion to how certain it is. Then ask the room to take two limits — $n\to\infty$ washes the prior out entirely and MMSE $\to$ MLE; $n = 0$ returns the prior untouched. Those limits are what convince students the formula is sensible rather than arbitrary.

**Then the connection that pays off the whole curriculum.** This is the [Kalman gain](../../Intro_Time_Series/Intro_AdFilt_KF.ipynb), identically. Signal variance over total variance, weighting each source by its certainty. The Kalman filter *is* Bayesian estimation for linear-Gaussian models, run recursively — and it also matches the Wiener filter's $S_d/(S_d+S_v)$ from [Statistical SP](../../Intro_DSP/Statistical_Signal_Processing.ipynb), which is the same trust dial indexed by frequency instead of time. Three workshops, one formula. Saying that explicitly is one of the highest-value minutes in this track.

**Handle the prior objection head-on, because someone will raise it.** "Isn't choosing a prior subjective?" Yes — and the demo answers it empirically: by $n = 200$ the posterior has forgotten the prior almost completely. With enough data the prior washes out; with little data it *should* matter, because with little data you genuinely do have to rely on prior knowledge. The honest risk is a *confident wrong* prior with little data, which is a real failure mode worth naming rather than dismissing.

**On the demo.** Point at both things the posterior does as $n$ grows: it **slides** from the prior mean toward the truth, and it **sharpens**. Those are the two Bayesian updates — location and confidence — and the Kalman filter does exactly these two things every timestep.
</details>

## 4. The Bayesian View

💡 **Intuition.** The frequentist asks 'what would happen over repeated experiments?' The Bayesian carries a **belief** about $\theta$ (a prior), and Bayes' rule updates it with data into a posterior — the same belief-update choreography as the [Kalman filter](../../Intro_Time_Series/Intro_AdFilt_KF.ipynb), which is exactly Bayesian estimation for linear-Gaussian models, run recursively.

**MMSE estimator.** Minimizing $E[(\hat\theta - \theta)^2]$ over all functions of the data gives $\hat\theta_{MMSE} = E[\theta \mid x]$ — the posterior mean. (Proof: expand the square around the conditional mean; the cross term vanishes by iterated expectation.)

**Worked example — Gaussian prior + Gaussian data.** Prior $\theta \sim \mathcal{N}(\mu_0, \tau^2)$, data $x_i \sim \mathcal{N}(\theta, \sigma^2)$. The posterior is Gaussian with
$$E[\theta \mid x] = \frac{\tau^2}{\tau^2 + \sigma^2/n} \, \bar{x} \;+\; \frac{\sigma^2/n}{\tau^2 + \sigma^2/n} \, \mu_0$$
— a **precision-weighted blend** of data and prior. Compare the Kalman gain: same formula, same trust dial. Little data ⇒ lean on the prior; lots of data ⇒ the prior washes out and MMSE → MLE.

In [4]:
# Watch the posterior sharpen and slide from prior to truth as data arrives
mu0, tau, sigma, theta_true = 0.0, 1.0, 2.0, 1.7
th = np.linspace(-2.5, 3.5, 500)
x_all = rng.normal(theta_true, sigma, 200)

plt.figure(figsize=(8, 3))
for n_obs, alpha in [(0, 0.35), (2, 0.5), (10, 0.7), (200, 1.0)]:
    if n_obs == 0:
        m, v = mu0, tau**2
    else:
        xb = x_all[:n_obs].mean()
        v = 1 / (1/tau**2 + n_obs/sigma**2)
        m = v * (mu0/tau**2 + n_obs*xb/sigma**2)
    plt.plot(th, np.exp(-(th-m)**2/(2*v)) / np.sqrt(2*np.pi*v), alpha=alpha, label=f"n={n_obs}")
plt.axvline(theta_true, color="k", linestyle="--", linewidth=0.8, label="truth")
plt.legend(); plt.title("Posterior evolution: prior → data-dominated")
plt.tight_layout(); plt.show()

/tmp/ipykernel_2014894/1447700788.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** Four posteriors, and they do **two** things at once as data arrives: they **slide** from the prior mean (0.0) toward the truth (1.7), and they **sharpen** from a broad prior into a narrow spike. Location and confidence, updated together — which is exactly what a Kalman filter does at every timestep.

**Read the $n = 0$ curve first.** It is the prior, unmodified: our belief before seeing anything. That the framework has a well-defined answer with *zero* data is the philosophical difference from Sessions 1–2, where MLE on an empty sample is undefined.

**Then the blend that produces the rest.** The posterior mean is
$$E[\theta \mid x] = \frac{\tau^2}{\tau^2 + \sigma^2/n}\,\bar x \;+\; \frac{\sigma^2/n}{\tau^2 + \sigma^2/n}\,\mu_0,$$
a **precision-weighted average** of the data and the prior — each source counted in proportion to how certain it is. Take the two limits and the formula justifies itself: as $n \to \infty$ the data's precision $n/\sigma^2$ dominates, the prior weight goes to zero, and MMSE converges to the MLE; at $n = 0$ it returns the prior untouched. The $n = 200$ curve in the plot is essentially the frequentist answer.

**And this is the Kalman gain.** Not analogous to it — *identically* it. Signal variance over total variance, weighting each source by its certainty. The [Kalman filter](../../Intro_Time_Series/Intro_AdFilt_KF.ipynb) is Bayesian estimation for linear-Gaussian models, run recursively, with the posterior from each step serving as the prior for the next. It is also the same formula as the Wiener filter's $S_d/(S_d+S_v)$ in [Statistical SP](../../Intro_DSP/Statistical_Signal_Processing.ipynb) — one trust dial, indexed by time in one workshop and by frequency in the other.

**What MMSE actually guarantees, which is stronger than it looks.** Minimising $E[(\hat\theta - \theta)^2]$ over **all** functions of the data — not just linear ones — gives the posterior mean. So $E[\theta \mid x]$ is not the best estimator in some restricted class; it is the best estimator, full stop, under squared-error loss. That optimality is why the Kalman filter can claim to be optimal rather than merely good.

**On the objection that priors are subjective.** They are, and the plot is the honest answer: by $n = 200$ the posterior has essentially forgotten the prior, so with enough data the choice stops mattering. With *little* data the prior matters a great deal — which is appropriate, because with little data you genuinely are relying on prior knowledge, and pretending otherwise does not make the reliance disappear. The real failure mode worth naming is a **confident wrong prior with little data**: a narrow $\tau$ centred in the wrong place will hold the posterior away from the truth for a long time. That is a modelling risk, not an argument against the framework.

---
### 🕐 Session 4 of 4 — *Sufficiency* (~30 min)
**Goal:** find the statistics that capture ALL the information; compress data without losing θ.
**Builds on:** Sessions 1–3.

---

<details>
<summary>🎓 <b>Teacher notes — Session 4: Sufficiency</b></summary>

**Timing (~30 min).** 8 min the compression framing · 8 min Fisher–Neyman · 8 min the demo · 6 min Rao–Blackwell and the Kalman payoff.

**Board first — frame it as lossless compression, because that is what it is.** You have 1000 samples. Can you throw away 999 numbers and keep one, without losing anything *about $\mu$*? For Gaussian data with known $\sigma$, yes — the sample mean is **sufficient**, and the individual samples carry no further information about $\mu$ once you know it. Call it "the ultimate lossy compression, with zero loss for the question asked." Students find that phrasing sticky, and the qualifier matters: the discarded data still carries information about *other* questions, like whether the Gaussian assumption holds at all.

**Fisher–Neyman is the practical tool, and it is mechanical.** $p(x;\theta) = g(T(x),\theta)\,h(x)$: if the density factors so that $\theta$ meets the data *only* through $T$, then $T$ is sufficient. Work the Gaussian example on the board — the exponent contains $\sum_i x_i$ and $\sum_i x_i^2$ and nothing else that touches $\mu$ — so you can *read off* sufficient statistics rather than verifying a conditional distribution.

**Ask the room.** "What is sufficient for a Bernoulli?" The count of successes. "For a uniform on $[0,\theta]$?" The maximum. That second one is worth doing because it is not an average, which breaks the pattern students start to assume.

**Be clear about what the demo does and does not show.** It compares the first-sample estimator (variance 2.29) with the sample mean (0.0896) — a factor of 25.5, which is exactly $n = 25$. That is *consistent with* Rao–Blackwell's spirit, and it is not a proof of the theorem; the code comment says "empirical check of the spirit," and that honesty is worth preserving when you present it. The actual theorem says conditioning any unbiased estimator on a sufficient statistic never increases variance, and $E[x_1 \mid \bar x] = \bar x$ is the specific instance that turns one estimator into the other.

**The factor being exactly $n$ is the point worth extracting.** Using one sample instead of 25 costs precisely a factor of 25 in variance — sufficiency is not a marginal improvement, it is the difference between using your data and discarding it. Have the room predict the ratio before revealing it.

**Close with the payoff that ties the whole workshop together.** The [Kalman filter](../../Intro_Time_Series/Intro_AdFilt_KF.ipynb) carries only a mean and a covariance, no matter how many measurements it has processed — and it loses nothing, because for linear-Gaussian models those two numbers are a sufficient statistic for the entire history. That is why recursive estimation is possible at all: without sufficiency you would have to store every measurement forever. Sessions 1–4 have quietly been building the justification for the filter this curriculum uses everywhere.
</details>

## 5. Sufficient Statistics

💡 **Intuition.** Sometimes a summary of the data is *lossless for the parameter*: once you know the sample mean of Gaussian data, the individual samples carry no further information about $\mu$. That summary is a **sufficient statistic** — the ultimate lossy compression with zero loss *for the question asked*. It's why the Kalman filter can carry just a mean and covariance instead of all past data.

**Definition.** $T(x)$ is sufficient for $\theta$ if the conditional distribution of the data given $T$ does not depend on $\theta$.

**Fisher–Neyman factorization.** $T$ is sufficient iff $p(x; \theta) = g(T(x), \theta) \, h(x)$ — the density splits into a part that sees $\theta$ only through $T$, and a $\theta$-free part.

**Example (Gaussian, known σ).** $\prod_i p(x_i;\mu) \propto \exp\big( \frac{\mu}{\sigma^2} \sum_i x_i - \frac{n\mu^2}{2\sigma^2}\big) \cdot h(x)$ — $\theta$ meets the data only through $\sum_i x_i$. The sample sum (equivalently mean) is sufficient. **Rao–Blackwell** (stated): conditioning any unbiased estimator on a sufficient statistic never increases variance — good estimators should *only* depend on $T$.

In [5]:
# Empirical check of Rao–Blackwell's spirit: an estimator ignoring T is beatable
# "First sample only" is unbiased for μ but wasteful; the mean (a function of T) dominates it
X = rng.normal(2.0, 1.5, size=(20000, 25))
print(f"first-sample estimator:  var {X[:, 0].var():.4f}")
print(f"sample-mean estimator:   var {X.mean(1).var():.4f}   ← function of the sufficient statistic")

first-sample estimator:  var 2.2865
sample-mean estimator:   var 0.0896   ← function of the sufficient statistic


**What just happened.** Two unbiased estimators of the same mean: using only the first sample gives variance **2.2865**; using the sample mean gives **0.0896**. The ratio is **25.5** — which is exactly $n = 25$.

**That the factor is precisely $n$ is the point.** Both estimators are unbiased, so neither is *wrong*; one simply discards 24 of its 25 data points. And the price of discarding them is the full factor $n$ in variance, because the sample mean's variance is $\sigma^2/n$ while a single observation's is $\sigma^2$. Sufficiency is not a marginal refinement — it is the difference between using your data and throwing it away.

Note also that the first-sample variance, 2.2865, is essentially $\sigma^2 = 2.25$ as it must be. Both numbers check out against theory rather than merely against each other.

**What sufficiency means, framed as compression.** A statistic $T(x)$ is sufficient for $\theta$ if the conditional distribution of the data given $T$ does not depend on $\theta$ — informally, once you know $T$, the individual samples tell you nothing further *about $\theta$*. For Gaussian data with known $\sigma$, the sample sum (equivalently the mean) is sufficient: you can discard 999 of 1000 numbers and lose nothing about $\mu$.

The qualifier matters and is worth stating: **nothing about $\mu$**. The discarded samples still carry information about other questions — whether the data is really Gaussian, whether there are outliers, whether the variance is what you assumed. Sufficiency is lossless *for the question asked*, and model checking is a different question.

**Fisher–Neyman makes it mechanical.** If $p(x;\theta) = g(T(x),\theta)h(x)$ — the density factors so $\theta$ meets the data only through $T$ — then $T$ is sufficient. For the Gaussian, the exponent contains $\sum_i x_i$ and nothing else that touches $\mu$, so you can *read the sufficient statistic off the density* rather than verifying a conditional distribution.

**Be precise about what this cell demonstrates.** It shows the mean beating a wasteful estimator by exactly the factor sufficiency predicts. That is consistent with **Rao–Blackwell** — conditioning any unbiased estimator on a sufficient statistic never increases variance — but it is not a proof of it, and the code comment's "empirical check of the spirit" is the honest description. The theorem's specific content here is that $E[x_1 \mid \bar x] = \bar x$: conditioning the first-sample estimator on the sufficient statistic *turns it into* the sample mean.

**And the payoff ties the workshop to the rest of the curriculum.** The [Kalman filter](../../Intro_Time_Series/Intro_AdFilt_KF.ipynb) carries only a mean and a covariance regardless of how many measurements it has processed — and loses nothing, because for linear-Gaussian models those two numbers are a sufficient statistic for the *entire measurement history*. Without sufficiency, recursive estimation would be impossible: you would have to store every observation forever and re-process it at each step. Four sessions of estimation theory turn out to be the justification for the filter running in every GPS receiver.

## 6. Conclusion

MLE turns densities into losses; CRLB is the floor and Fisher information the currency; Bayes blends prior and data by precision; sufficiency tells you what to keep. You can now *read* the estimation claims in every filtering and ML paper.

---
## Where next

- [Adaptive Filtering: Kalman](../../Intro_Time_Series/Intro_AdFilt_KF.ipynb) — recursive MMSE with Gaussian everything.
- [Statistical Signal Processing](../../Intro_DSP/Statistical_Signal_Processing.ipynb) — detection: estimation's sibling.
- [Uncertainty in ML](../../Intro_Mach_Learn/Uncertainty_in_ML.ipynb) — what happens to these guarantees under deep models.